# Usando o modelo treinado

Carrega o modelo e o tokenizer salvos pelo notebook 2 e gera texto no estilo dos papers do arXiv.
Não treina nada aqui.

In [1]:
import json

import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import tokenizer_from_json

I0000 00:00:1792028138.235536  451581 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792028138.318903  451581 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792028141.352651  451581 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
modelo = load_model("data/modelo/modelo_lm.keras")

with open("data/modelo/tokenizer.json", encoding="utf-8") as f:
    tokenizer = tokenizer_from_json(f.read())

with open("data/modelo/config.json", encoding="utf-8") as f:
    config = json.load(f)

SEQ_LEN = config["seq_len"]
print("arquitetura:", config["arquitetura"], "| janela:", SEQ_LEN)

# id -> palavra (o contrário do word_index)
inverso = {i: p for p, i in tokenizer.word_index.items()}

E0000 00:00:1792028143.135942  451581 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


arquitetura: gru | janela: 20


In [3]:
def gerar(seed_text, n_palavras=40, temperatura=0.8):
    texto = seed_text.lower()
    for _ in range(n_palavras):
        ids = tokenizer.texts_to_sequences([texto])[0]
        ids = pad_sequences([ids], maxlen=SEQ_LEN, padding="pre")
        probs = modelo.predict(ids, verbose=0)[0]

        if temperatura == 0:
            escolhida = int(np.argmax(probs))
        else:
            p = np.log(probs + 1e-9) / temperatura
            p = np.exp(p - p.max())
            p /= p.sum()
            escolhida = int(np.random.choice(len(p), p=p))

        palavra = inverso.get(escolhida)
        if palavra is None or palavra == "<OOV>":
            continue
        texto += " " + palavra
    return texto

In [4]:
seeds = [
    "in this paper we propose",
    "experimental results show that",
    "the proposed method outperforms",
]
for s in seeds:
    print(gerar(s, 40, 0.8), "\n")

in this paper we propose a second funding the selector which does not provide evidence which outside a interface of the current output every response is the evidence that it uses generated by instead of abcagent and a model bound by ht one graphs 

experimental results show that the evidence to its derive the final answer it applies both run to two evidence and practical reasoning through a negative damped j factual response action response and attribution the detection of the backbone block diffusion language model 

the proposed method outperforms the highest task constant is therefore used to the position of each log annotator tasks when poorly needed to accommodate the training and selected drafts in many edge strategically updating and two benchmarks but kept verify for minimal inference 



In [5]:
# escreve a sua seed (precisa ser em inglês) e ajusta a temperatura
print(gerar("we introduce a novel", 60, 0.9))

we introduce a novel system and clinicopathologic execution rerun shows that them whose evaluator rmt rubrics should often enable data in agent through any query opposed with written evidence into the current trajectory with an explicit case to locate most route assuming every online quantification in autoregressive operations and side model weights that does not reliably restart across the exceptional context and set
